# 1.

In [2]:
import ast
import io
import json
import os
import re
import shutil
import subprocess
import sys
import stat
import time
import tokenize
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Formatter untuk standarisasi kode Python
import autopep8
import black

# Engine export Excel untuk pandas
import openpyxl

# Progress bar & notebook display
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# Waktu
run_time = datetime.now()

# Konfigurasi visualisasi default
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

print("=" * 65)
print(f"{'LIBRARY INITIALIZATION':^65}")
print("-" * 65)
print(f"Python     : {sys.version.split()[0]}")
print(f"✅ {run_time.strftime('%Y-%m-%d %H:%M:%S')} - Libraries loaded successfully.")
print("-" * 65)

                     LIBRARY INITIALIZATION                      
-----------------------------------------------------------------
Python     : 3.10.6
✅ 2026-05-26 01:42:42 - Libraries loaded successfully.
-----------------------------------------------------------------


Konfigurasi dan Helper


In [15]:
# ==============================================================================
# DIRECTORY CONFIGURATION & INITIALIZATION
# Menentukan path utama, struktur folder dataset, dan file output
# ==============================================================================

# Root directory dan file input
BASE_DIR = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code"
DATASET_FOLDER = "dataset(2)"
OUTPUT_FOLDER = "output(2)"

INPUT_GITHUB = os.path.join(BASE_DIR, "asli", "nim_github.txt")

# Struktur folder pipeline
DIRS = {
    "DATASET"      : os.path.join(BASE_DIR, DATASET_FOLDER),

    # Preprocessing
    "RAW"          : os.path.join(BASE_DIR, DATASET_FOLDER, "01_Raw"),
    "NORM"         : os.path.join(BASE_DIR, DATASET_FOLDER, "02_Normalized"),
    "CONV"         : os.path.join(BASE_DIR, DATASET_FOLDER, "03_Converted"),
    "CLEAN"        : os.path.join(BASE_DIR, DATASET_FOLDER, "04_Cleaned"),

    # Formatter experiment
    "AUTOPEP8"     : os.path.join(BASE_DIR, DATASET_FOLDER, "05a_Autopep8"),
    "BLACK"        : os.path.join(BASE_DIR, DATASET_FOLDER, "05b_Black"),
    
    "FILTERED"     : os.path.join(BASE_DIR, DATASET_FOLDER, "05c_Filtered"),

    # AST & Graph
    "AST"          : os.path.join(BASE_DIR, DATASET_FOLDER, "06_AST"),
    "AST_VISUAL"   : os.path.join(BASE_DIR, DATASET_FOLDER, "06a_AST_visual"),
    "GRAPH"        : os.path.join(BASE_DIR, DATASET_FOLDER, "07_Graph"),
    "INPUT_GRAPH"  : os.path.join(BASE_DIR, DATASET_FOLDER, "08_Graph2vec_Input"),
    "EMBEDDING"    : os.path.join(BASE_DIR, DATASET_FOLDER, "09_Graph2vec_Embedding"),

    # Output
    "LOGS"         : os.path.join(BASE_DIR, OUTPUT_FOLDER),
    "RUNTIME_A"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_autopep8"),
    "RUNTIME_B"    : os.path.join(BASE_DIR, OUTPUT_FOLDER, "run_black"),
    # folder output untuk skor kemiripan (COSINE SIMILARITY)
    "SIMILARITY"   : os.path.join(BASE_DIR, OUTPUT_FOLDER,"similarity"),
}

# File output penelitian
RESULTS = {
    # Preprocessing
    "CLONE_REPORT"    : os.path.join(DIRS["LOGS"], "01_clone_report.xlsx"),
    "NORM_REPORT"     : os.path.join(DIRS["LOGS"], "02_normalization_report.xlsx"),
    "CONV_REPORT"     : os.path.join(DIRS["LOGS"], "03_conversion_report.xlsx"),
    "CLEAN_REPORT"    : os.path.join(DIRS["LOGS"], "04_cleaning_report.xlsx"),

    # Formatter
    "ERR_AUTOPEP"     : os.path.join(DIRS["LOGS"], "05a_autopep_errors.json"),
    "ERR_BLACK"       : os.path.join(DIRS["LOGS"], "05b_black_errors.json"),

    # Statistik kode
    "LOC_REPORT"      : os.path.join(DIRS["LOGS"], "06_loc_report.xlsx"),

    # Runtime Autopep8
    "RUN_PROJECT_A"   : os.path.join(DIRS["RUNTIME_A"], "07a_runtime_project_AUTOPEP8.xlsx"),
    "RUN_FUNCTION_A"  : os.path.join(DIRS["RUNTIME_A"], "07b_runtime_function_AUTOPEP8.xlsx"),
    "RUN_COMPARE_A"   : os.path.join(DIRS["RUNTIME_A"], "07c_runtime_compare_AUTOPEP8.xlsx"),

    # Runtime Black
    "RUN_PROJECT_B"   : os.path.join(DIRS["RUNTIME_B"], "07a_runtime_project_BLACK.xlsx"),
    "RUN_FUNCTION_B"  : os.path.join(DIRS["RUNTIME_B"], "07b_runtime_function_BLACK.xlsx"),
    "RUN_COMPARE_B"   : os.path.join(DIRS["RUNTIME_B"], "07c_runtime_compare_BLACK.xlsx"),

    # Submission
    "SUBMISSION"      : os.path.join(DIRS["LOGS"], "submission_report.xlsx"),

    # AST & Graph
    "EXTRACT_AST"     : os.path.join(DIRS["LOGS"], "09a_AST_report.xlsx"),
    "CONSTRUCT_GRAPH" : os.path.join(DIRS["LOGS"], "09b_Graph_report.xlsx"),
    "LIST_GRAPH"      : os.path.join(DIRS["LOGS"], "09c_List_Graph_report.xlsx"),

    # Graph2Vec
    "EMBEDDING_REPORT": os.path.join(DIRS["LOGS"], "10_embedding_report.xlsx"),
    "EMBEDDING_VECTOR": os.path.join(DIRS["LOGS"], "10a_embedding_vector.xlsx"),

    # Similarity
    "SIMILARITY"      : os.path.join(DIRS["LOGS"], "11_similarity_report.xlsx"),
    "SIMILARITY_MODUL" : os.path.join(DIRS['LOGS'], "11a_Similarity_per_Modul.xlsx"),
    "SIMILARITY_SUMMARY" : os.path.join(DIRS['LOGS'], "11b_similarity_summary.xlsx"),
    
    # Euclidean Similarity
    "EUCLIDEAN" : os.path.join(DIRS['LOGS'], "12_euclidean_similarity.xlsx"),
    "EUCLIDEAN_MODUL" : os.path.join(DIRS['LOGS'], "12a_euclidean_similarity_modul.xlsx"),
    
    "SIMILARITY_COMPARE" : os.path.join(DIRS['LOGS'], "13_similarity_comparison.xlsx"),
    
    # METRICS
    "METRICS"         : os.path.join(DIRS["LOGS"], "metrics_evaluation_report.xlsx"),
}

# VALIDATION & DIRECTORY INITIALIZATION
print("=" * 70)
print(f"{'DIRECTORY CONFIGURATION':^70}")
print("=" * 70)

# Validasi root directory
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"❌ BASE_DIR tidak ditemukan: {BASE_DIR}")

# Validasi file input
if not os.path.isfile(INPUT_GITHUB):
    raise FileNotFoundError(f"❌ File input tidak ditemukan: {INPUT_GITHUB}")

if not INPUT_GITHUB.endswith(".txt"):
    raise ValueError("❌ File input harus berekstensi .txt")

if os.path.getsize(INPUT_GITHUB) == 0:
    raise ValueError(f"❌ File input kosong: {INPUT_GITHUB}")

# Membuat folder pipeline
folders_created = 0

for name, path in DIRS.items():
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        folders_created += 1
        status = "[NEW]"
    elif not os.path.isdir(path):
        raise NotADirectoryError(f"❌ Path bukan folder: {path}")
    else:
        status = "[EXISTS]"
    print(f"{status:<10} {name:<12} : {path}")

# Validasi parent folder output
missing_results_parent = []

for name, path in RESULTS.items():
    parent_dir = os.path.dirname(path)
    if not os.path.exists(parent_dir):
        missing_results_parent.append(
            f"{name} : {parent_dir}"
        )

if missing_results_parent:
    raise FileNotFoundError(
        "❌ Parent folder RESULTS tidak ditemukan:\n"
        + "\n".join(missing_results_parent)
    )

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("-" * 70)
print(f"Folders Created  : {folders_created}")
print(f"Dataset Root     : {DIRS['DATASET']}")
print(f"Input File       : {INPUT_GITHUB}")
print("=" * 70)
print(f"Diproses pada {timestamp}")

                       DIRECTORY CONFIGURATION                        
[EXISTS]   DATASET      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)
[EXISTS]   RAW          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\01_Raw
[EXISTS]   NORM         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\02_Normalized
[EXISTS]   CONV         : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\03_Converted
[EXISTS]   CLEAN        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\04_Cleaned
[EXISTS]   AUTOPEP8     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05a_Autopep8
[EXISTS]   BLACK        : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05b_Black
[EXISTS]   FILTERED     : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05c_Filtered
[EXISTS]   AST          : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_AST
[EXISTS]   AST_VISUAL   : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06a_AST_visual
[EXISTS]   GRAPH        : D:\PUTRI\D4\SEMESTER

In [4]:
# HELPER FUNCTION Hitung File dalam dataset

def count_all_files(dataset_path, extensions=None, exclude_dirs=None):
    total_files = 0
    by_extension = {}

    exclude_dirs = set(exclude_dirs or [])

    for root, dirs, files in os.walk(dataset_path):
        # skip folder tertentu
        dirs[:] = [d for d in dirs if d not in exclude_dirs]

        for file in files:
            ext = os.path.splitext(file)[1].lower()

            # filter ekstensi jika diberikan
            if extensions and ext not in extensions:
                continue

            total_files += 1
            by_extension[ext] = by_extension.get(ext, 0) + 1

    return {
        "total_files": total_files,
        "by_extension": by_extension
    }

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-05-26 01:42:43


In [5]:
# ==============================================================================
# METRIC EVALUATION HELPERS
# ==============================================================================
def save_stage_evaluation(report_path, stage_name, total_files, total_size_kb, execution_time, total_loc=0):
    """Simpan atau perbarui evaluasi makro per tahap ke sheet 'stage_level_evaluation'."""
    SHEET = "stage_level_evaluation"
    avg_size = (total_size_kb / total_files if total_files > 0 else 0)
    avg_loc = (total_loc / total_files if total_files > 0 else 0)
    
    new_row = {
        "Tahap": stage_name, "Total File": total_files, "Total Size (KB)": round(total_size_kb, 2),
        "Avg Size per File (KB)": round(avg_size, 2), "Avg LOC per File": round(avg_loc, 2),
        "Waktu Eksekusi (s)": round(execution_time, 2)
    }

    # Gunakan dictionary untuk membaca seluruh sheet yang ada agar aman dari korup data
    all_sheets = {}
    if os.path.exists(report_path):
        try:
            with pd.ExcelFile(report_path, engine="openpyxl") as xls:
                for sheet in xls.sheet_names:
                    all_sheets[sheet] = pd.read_excel(xls, sheet_name=sheet)
        except Exception:
            pass

    # Ambil data lama khusus untuk sheet ini atau buat baru
    df = all_sheets.get(SHEET, pd.DataFrame())

    if not df.empty and "Tahap" in df.columns and stage_name in df["Tahap"].values:
        mask = df["Tahap"] == stage_name
        df.loc[mask, ["Total File", "Total Size (KB)", "Avg Size per File (KB)", "Avg LOC per File", "Waktu Eksekusi (s)"]] = [
            total_files, round(total_size_kb, 2), round(avg_size, 2), round(avg_loc, 2), round(execution_time, 2)
        ]
    else:
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    
    all_sheets[SHEET] = df

    # Tulis ulang seluruh sheet secara aman (Menghindari bug mode='a' openpyxl)
    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        for sheet, df_sheet in all_sheets.items():
            df_sheet.to_excel(writer, sheet_name=sheet, index=False)


def save_bulk_detail_metrics(report_path, sheet_name, metrics_list, stage_name):
    """
    Menyimpan metrik mikro secara massal (Bulk) untuk menghemat I/O Disk.
    metrics_list berisi list of dict: [{'nim':..., 'modul':..., 'file':..., 'value':...}]
    """
    if not metrics_list:
        return

    identity_cols = ["nim", "modul", "file_id"]
    
    # Read semua sheet lama agar data di sheet lain tidak hilang
    all_sheets = {}
    if os.path.exists(report_path):
        try:
            with pd.ExcelFile(report_path, engine="openpyxl") as xls:
                for sheet in xls.sheet_names:
                    all_sheets[sheet] = pd.read_excel(xls, sheet_name=sheet)
        except Exception:
            pass

    df = all_sheets.get(sheet_name, pd.DataFrame(columns=identity_cols))
    # Pastikan tipe data kolom identitas konsisten menjadi string agar pencocokan mask akurat
    for col in identity_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    # Pastikan kolom stage_name sudah tersedia
    if stage_name not in df.columns:
        df[stage_name] = None
        
    df_batch = pd.DataFrame(metrics_list)
    # Bersihkan whitespace dan seragamkan tipe data batch baru ke string
    for col in identity_cols:
        df_batch[col] = df_batch[col].astype(str).str.strip()
    
    # Amankan batch jika di dalam list RAM itu sendiri ada record duplikat (ambil yang terakhir)
    df_batch = df_batch.drop_duplicates(subset=identity_cols, keep='last')

    # Lakukan sinkronisasi data dari memori RAM ke DataFrame
    for item in metrics_list:
        nim, modul, file_name, value, file_id = item['nim'], item['modul'], item['file'], item['value'], item['file_id']
        mask = (df["nim"] == nim) & (df["modul"] == modul) & (df["file_id"] == file_id)
        
        if mask.any():
            df.loc[mask, stage_name] = round(value, 4)
        else:
            new_row = {"nim": str(nim), "modul": modul, "file": file_name, "file_id": file_id, stage_name: round(value, 4)}
            new_df = pd.DataFrame([new_row])
            # Hindari FutureWarning pandas
            if df.empty:
                df = new_df.copy()
            else:
                for col in df.columns:
                    if col not in new_df.columns:
                        new_df[col] = None

                for col in new_df.columns:
                    if col not in df.columns:
                        df[col] = None

                df = pd.concat([df, new_df[df.columns]], ignore_index=True)

    all_sheets[sheet_name] = df

    # Flush massal ke berkas Excel tunggal
    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        for sheet, df_sheet in all_sheets.items():
            df_sheet.to_excel(writer, sheet_name=sheet, index=False)
            # Mengatur kelebaran kolom otomatis untuk semua sheet yang aktif
            worksheet = writer.sheets[sheet]
            for col in worksheet.columns:
                max_len = max(len(str(cell.value or '')) for cell in col)
                worksheet.column_dimensions[col[0].column_letter].width = max(max_len + 3, 20)


def get_path_size(path, unit="kb"):
    """Menghitung kapasitas memori file/folder secara rekursif."""
    if not os.path.exists(path):
        return 0

    total_size = 0
    if os.path.isfile(path):
        try: total_size = os.path.getsize(path)
        except Exception: return 0
    else:
        for root, _, files in os.walk(path):
            for file in files:
                file_path = os.path.join(root, file)
                try:
                    if os.path.exists(file_path): total_size += os.path.getsize(file_path)
                except Exception: pass

    unit = unit.lower()
    factors = {"byte": 1, "kb": 1024, "mb": 1024**2, "gb": 1024**3}
    if unit in factors:
        return total_size if unit == "byte" else round(total_size / factors[unit], 4)
    else:
        raise ValueError("Unit tidak valid. Gunakan: byte/kb/mb/gb")


def count_loc(file_path):
    """Menghitung kuantitas baris kode aktif (tanpa baris kosong)."""
    if not os.path.exists(file_path):
        return 0

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".py":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                return sum(1 for line in f if line.strip())
        elif ext == ".ipynb":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                notebook = json.load(f)
            loc = 0
            for cell in notebook.get("cells", []):
                if cell.get("cell_type") == "code":
                    source = cell.get("source", [])
                    loc += sum(1 for line in source if str(line).strip())
            return loc
        return 0
    except Exception:
        return 0

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Function dijalankan pada {timestamp}")

✅ Function dijalankan pada 2026-05-26 01:42:43


# B. AST

## 1. Ekstraksi AST

### Ekstraks AST (py -> JSON)

In [6]:
# FUNCTION HELPER - AST EXTRACTION
# ==============================================================================

# Filter file berdasarkan requirement modul
def filter_files_by_requirement(py_files, input_dir, req_dict):
    filtered = []

    for file_path in py_files:
        rel_path = os.path.relpath(file_path, input_dir)
        parts = rel_path.split(os.sep)

        if len(parts) < 3:
            continue

        nim, modul, filename = parts[0], parts[1], parts[-1]
        if modul in req_dict and filename in req_dict[modul]:
            filtered.append(file_path)

    return filtered

# Function untuk logging
def save_log(nim, modul, file, metrics=None, status="SUCCESS", message=""):
    m = metrics if metrics else {
        "depth": 0, "node_count": 0, "function_count": 0, 
        "variable_count": 0, "cyclomatic_complexity": 0, "loc": 0
    }

    return {
        "nim": nim,
        "modul": modul,
        "file": file,
        "kedalaman_ast": m["depth"],
        "jumlah_node": m["node_count"],
        "jumlah_function": m["function_count"],
        "jumlah_variabel_unik": m["variable_count"],
        "cyclomatic_complexity": m["cyclomatic_complexity"],
        "loc": m["loc"],
        "status": status,
        "keterangan": message
    }
    
# READ FILE
# membaca isi file .py
# ------------------------------------------------------------------------------
def read_python_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read(), None
    except Exception as e:
        return None, str(e)

# PARSE AST 
# mengubah kode ke bentuk AST (objek), dan cek jika syntax error
# ------------------------------------------------------------------------------
def parse_ast(code):
    try:
        tree = ast.parse(code)
        return tree, None, None
    except SyntaxError as e:
        return None, "SYNTAX_ERROR", f"SyntaxError: {str(e)}"
    except Exception as e:
        return None, "FAILED", str(e)

# AST TO DICT (RECURSIVE)
# mengubah objek AST ke dict untuk nantinya disimpan ke JSON
# ------------------------------------------------------------------------------
def ast_to_dict(node):
    if isinstance(node, ast.AST):
        result = {"type": type(node).__name__}
        for field, value in ast.iter_fields(node):
            result[field] = ast_to_dict(value)
        return result
    elif isinstance(node, list):
        return [ast_to_dict(item) for item in node]
    else:
        # HANDLE NON-SERIALIZABLE TYPES
        if node is Ellipsis:
            return "Ellipsis"

        if isinstance(node, (str, int, float, bool)) or node is None:
            return node
        
        return str(node)

# METRICS EXTRACTION
# menghitung kedalaman AST, jumlah node, jumlah function, jumlah variabel
# ------------------------------------------------------------------------------
def compute_ast_metrics(tree, code):
    max_depth = 0
    node_count = 0
    function_count = 0
    cc_score = 1
    unique_vars = set() # Menggunakan set untuk menyimpan nama unik

    # Daftar node yang menambah Cyclomatic Complexity
    # (if, while, for, except, with, logical operators)
    cc_nodes = (
        ast.If, ast.While, ast.For, ast.AsyncFor, 
        ast.ExceptHandler, ast.With, ast.AsyncWith,
        ast.IfExp, ast.Assert
    )
    
    def visit(node, depth=0):
        nonlocal max_depth, node_count, function_count, cc_score

        if isinstance(node, ast.AST):
            node_count += 1
            max_depth = max(max_depth, depth)
            
            # Hitung Cyclomatic Complexity
            if isinstance(node, cc_nodes):
                cc_score += 1
            # Tambahan untuk BoolOp (and/or bisa punya banyak values)
            if isinstance(node, ast.BoolOp):
                cc_score += len(node.values) - 1

            # hitung function
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda)): 
                function_count += 1

            # hitung variabel unik
            if isinstance(node, ast.Name):
                if isinstance(node.ctx, (ast.Store, ast.Param)):
                    unique_vars.add(node.id)

            for child in ast.iter_child_nodes(node):
                visit(child, depth + 1)

    visit(tree)
    
    # hitung loc
    # loc = len(code.split('\n'))
    loc = sum(1 for line in code.split('\n') if line.strip())
    
    return {
        "depth": max_depth,
        "node_count": node_count,
        "function_count": function_count,
        "variable_count": len(unique_vars), # Jumlah elemen unik di set
        "cyclomatic_complexity": cc_score,
        "loc": loc
    }

# SAVE JSON
# menyimpan hasil ekstraksi AST ke JSON
# ------------------------------------------------------------------------------
def save_json(data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# EXTRACT METADATA (nim, modul, filename)
# ------------------------------------------------------------------------------
def extract_metadata(root_dir, filepath):
    relative_path = os.path.relpath(filepath, root_dir)
    parts = relative_path.split(os.sep)

    nim = parts[0] if len(parts) > 0 else "unknown"
    modul = parts[1] if len(parts) > 1 else "unknown"
    filename = os.path.basename(filepath)

    return nim, modul, filename

# collect files python
def collect_py_files(input_dir):
    py_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".py"):
                py_files.append(os.path.join(root, file))
    return py_files

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Helper functions dijalankan pada {timestamp}")

✅ Helper functions dijalankan pada 2026-05-26 01:42:44


In [8]:
# RUN & EXECUTION
# ==============================================================================

def process_file(file_path, input_dir, output_dir, runtime_batch, size_batch, loc_batch):
    file_start_time = time.time()
    nim, modul, filename = extract_metadata(input_dir, file_path)
    file_id = os.path.splitext(filename)[0]
    code, read_err = read_python_file(file_path)
    if read_err:
        return save_log(nim, modul, filename, status="FAILED", message=read_err)

    tree, err_type, msg = parse_ast(code)

    if err_type:
        return save_log(nim, modul, filename, status=err_type, message=msg)

    try:
        metrics = compute_ast_metrics(tree, code)
        # AST → JSON
        ast_dict = ast_to_dict(tree)
        json_content = {
            "nim": nim,
            "modul": modul,
            "filename": filename,
            "metrics": {
                "kedalaman_ast": metrics["depth"],
                "jumlah_node": metrics["node_count"],
                "jumlah_function": metrics["function_count"],
                "jumlah_variabel_unik": metrics["variable_count"],
                "cyclomatic_complexity": metrics["cyclomatic_complexity"],
                "loc": metrics["loc"]
            },
            "ast": ast_dict  # Struktur AST Utama
        }
        relative_path = os.path.relpath(file_path, input_dir)
        output_path = os.path.join(output_dir, relative_path.replace(".py", ".json"))
        save_json(json_content, output_path)
        
        file_runtime = time.time() - file_start_time
        json_size = get_path_size(output_path, unit='kb')
        file_loc = count_loc(file_path)
        # Enkapsulasi basis data identitas berkas ke dalam objek kamus tunggal
        base_data = {"nim": nim, "modul": modul, "file": filename, "file_id": file_id}
        # Alokasi metrik ke penampung batch menggunakan teknik dictionary unpacking
        runtime_batch.append({**base_data, "value": file_runtime})
        size_batch.append({**base_data, "value": json_size})
        loc_batch.append({**base_data, "value": file_loc})

        return save_log(nim, modul, filename, metrics=metrics)
    except Exception as e:
        return save_log(nim, modul, filename, status="FAILED", message=str(e))
    
# EXECUTION
# --------------------------
print("="*60)
print(f"{' EXTRACT AST ':^60}")
print("-"*60)

INPUT_DIR = DIRS['FILTERED']
OUTPUT_DIR = DIRS['AST']

print(f"{'Dataset input':<30}: {INPUT_DIR}")

# STEP 1: scan untuk dapat requirement
# df_raw = scan_directory(INPUT_DIR)
# df_raw = df_raw[df_raw['modul'].str.lower() != 'unclassified']

# req_dict = extract_requirements_from_dataset(df_raw)

# STEP 2: ambil semua file
all_py_files = collect_py_files(INPUT_DIR)

# STEP 3: filter hanya requirement
# filtered_py_files = filter_files_by_requirement(all_py_files, INPUT_DIR, req_dict)

print(f"{'Total File input':<30}: {len(all_py_files)}")
# print(f"{'Total File setelah filter':<30}: {len(filtered_py_files)}")

logs = []
success = 0
failed = 0
syntax_err = 0

extract_start = time.time()
runtime_batch = []
size_batch = []
loc_batch = []
total_loc = 0

for file_path in tqdm(all_py_files, desc="Extracting AST", unit="file"):
    log = process_file(file_path, INPUT_DIR, OUTPUT_DIR, runtime_batch, size_batch, loc_batch)
    logs.append(log)

    # update counter
    if log["status"] == "SUCCESS":
        success += 1
        if "loc" in log:
            total_loc += log["loc"]
    elif log["status"] == "FAILED":
        failed += 1
    elif log["status"] == "SYNTAX_ERROR":
        syntax_err += 1

print(f"{'AST disimpan di':<30}: {OUTPUT_DIR}")
print(f"\n{' PROSES EXTRACTION SELESAI ':=^60}")

# SUMMARY & REPORTING
# --------------------------
# 1. Konversi logs ke DataFrame
df_logs = pd.DataFrame(logs)

# 2. Filter data, ambil yang statusnya sukses
df_success = df_logs[df_logs['status'] == 'SUCCESS']

# 3. Simpan Report ke Excel
report_path = RESULTS['EXTRACT_AST']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

# 4. Perhitungan Ringkasan Dasar (Logic Corrected)
total_input = len(all_py_files) # Total file .py di folder input
total_processed = len(df_logs)  # Total file yang masuk ke loop processing

# Success Rate dihitung dari (SUCCESS / TOTAL INPUT)
success_rate = (success / total_input * 100) if total_input > 0 else 0

extract_total_time = time.time() - extract_start
# Ekstraksi metrik mikro berkas secara massal (bulk) untuk sub-tahap AST
save_bulk_detail_metrics(RESULTS['METRICS'], "detail_runtime", runtime_batch, stage_name="AST")
save_bulk_detail_metrics(RESULTS['METRICS'], "detail_size", size_batch, stage_name="AST")
save_bulk_detail_metrics(RESULTS['METRICS'], "detail_loc", loc_batch, stage_name="AST")
# Sinkronisasi metrik makro kumulatif direktori untuk tahap 'AST Extraction'
save_stage_evaluation(
    report_path=RESULTS['METRICS'],
    stage_name="AST Extraction",
    total_files=count_all_files(DIRS['AST'])['total_files'],
    total_size_kb=get_path_size(DIRS['AST'], unit='kb'),
    total_loc=total_loc,
    execution_time=extract_total_time
)

display(HTML(f"<h3>RINGKASAN HASIL EKSTRAKSI AST</h3>"))
print("=" * 75)
print(f"{'Dataset Input':<35}: {INPUT_DIR}")
print(f"{'Total File Input (Dataset)':<35}: {total_input}")
print(f"{'Total File Diproses':<35}: {total_processed}")
print(f"{'Waktu Eksekusi':<35}: {extract_total_time:.2f} seconds")
print(f"{'Total File JSON disimpan':<35}: {count_all_files(DIRS['AST'])['total_files']}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal (FAILED)':<35}: {failed}")
print(f"{'Dilewati (SYNTAX ERROR)':<35}: {syntax_err}")
print(f"{'Success Rate':<35}: {success_rate:.2f}%")

print("\n" + "-" * 75)
print(f"Report Excel  : {report_path}")
print(f"Dataset JSON  : {OUTPUT_DIR}")

# Menampilkan 10 sampel teratas yang sukses
if not df_success.empty:
    print("\nSamples Data Sukses:")
    display(df_success.head(10))

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                        EXTRACT AST                         
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05c_Filtered
Total File input              : 2290


Extracting AST:   0%|          | 0/2290 [00:00<?, ?file/s]

AST disimpan di               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_AST

================ PROSES EXTRACTION SELESAI =================


Dataset Input                      : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\05c_Filtered
Total File Input (Dataset)         : 2290
Total File Diproses                : 2290
Waktu Eksekusi                     : 24.25 seconds
Total File JSON disimpan           : 2290
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 2290
Gagal (FAILED)                     : 0
Dilewati (SYNTAX ERROR)            : 0
Success Rate                       : 100.00%

---------------------------------------------------------------------------
Report Excel  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\09a_AST_report.xlsx
Dataset JSON  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_AST

Samples Data Sukses:


,nim,modul,file,kedalaman_ast,jumlah_node,jumlah_function,jumlah_variabel_unik,cyclomatic_complexity,loc,status,keterangan
0,2241720092,js02,p01.py,9,415,0,9,3,47,SUCCESS,
1,2241720092,js02,p02.py,9,113,0,1,1,12,SUCCESS,
2,2241720092,js02,p03.py,7,212,0,11,1,29,SUCCESS,
3,2241720092,js02,p04.py,5,142,0,4,1,18,SUCCESS,
4,2241720092,js02,tp.py,11,655,0,12,7,67,SUCCESS,
5,2241720092,js03,p01.py,13,501,0,3,1,86,SUCCESS,
6,2241720092,js03,p02.py,5,142,0,4,1,18,SUCCESS,
7,2241720092,js03,p03.py,7,224,0,4,1,22,SUCCESS,
8,2241720092,js03,p04.py,7,53,0,4,1,6,SUCCESS,
9,2241720092,js03,tp.py,11,1216,0,19,9,124,SUCCESS,


Diproses pada 2026-05-26 01:45:25


### Visualisasi AST

In [9]:
import os
import sys
import json
import ast
import traceback
import warnings
import logging
from tqdm.notebook import tqdm  # PERUBAHAN PENTING: Menggunakan versi auto/notebook untuk Jupyter
from datetime import datetime

# ===== PENCEGAHAN ANIMASI LOADING PECAH ==============================
# 1. Matikan peringatan standar
warnings.filterwarnings("ignore", category=UserWarning)

# 2. Matplotlib membandel karena menggunakan modul 'logging' C-level untuk peringatan Font.
# Kita matikan paksa akses loggernya di sini agar tidak merusak progress bar.
logging.getLogger('matplotlib.font_manager').disabled = True
logging.getLogger('matplotlib').setLevel(logging.ERROR)


# ===== IMPORT DEPENDENCIES ===========================================
try:
    import graphviz
    GRAPHVIZ_AVAILABLE = True
    print("✓ Graphviz tersedia")
except ImportError:
    GRAPHVIZ_AVAILABLE = False
    print("⚠️  Graphviz tidak tersedia, fallback ke matplotlib")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
    print("✓ Matplotlib tersedia")
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("⚠️  Matplotlib tidak tersedia")

# ===== KONFIGURASI =================================================
MAX_NODES_VIS = 80
MAX_DEPTH_VIS = 10

# Pastikan variabel DIRS sudah ada di cell environment Anda sebelumnya
INPUT_DIR      = DIRS['AST']
OUTPUT_VIS_DIR = DIRS['AST_VISUAL']

# ===== KUMPULKAN SEMUA .json =========================================
print("\n" + "="*70)
print("VALIDASI: VISUALISASI AST")
print("="*70)

all_ast_files = []
for nim in os.listdir(INPUT_DIR):
    nim_path = os.path.join(INPUT_DIR, nim)
    if not os.path.isdir(nim_path):
        continue
    for modul in os.listdir(nim_path):
        modul_path = os.path.join(nim_path, modul)
        if not os.path.isdir(modul_path):
            continue
        for fname in os.listdir(modul_path):
            if fname.endswith('.json'):
                fpath = os.path.join(modul_path, fname)
                all_ast_files.append((fpath, nim, modul, fname.replace('.json', '.py')))

print(f"✓ Input folder  : {INPUT_DIR}")
print(f"✓ Output folder : {OUTPUT_VIS_DIR}")
print(f"✓ Total AST JSON: {len(all_ast_files)} file")
print("="*70)

if len(all_ast_files) == 0:
    print("⚠️  Tidak ada file AST ditemukan!")
    print("   Jalankan cell AST extraction terlebih dahulu!")
    sys.exit()
else:
    print(f"✓ Siap visualisasi {len(all_ast_files)} file AST\n")


# ===== KONVERSI DICT → ast.AST OBJECT ================================
def is_valid_identifier(name):
    if not isinstance(name, str):
        return False
    if not name.isidentifier():
        return False
    if name.startswith('_'):
        return True
    return True

def dict_to_ast(d, path="root"):
    if not isinstance(d, dict):
        return None

    node_type_name = d.get('type')
    if not node_type_name or not isinstance(node_type_name, str):
        return None

    node_class = getattr(ast, node_type_name, None)
    if node_class is None or not (isinstance(node_class, type) and issubclass(node_class, ast.AST)):
        node = ast.AST()
        node._type_name = node_type_name
    else:
        node = node_class()

    for key, val in d.items():
        if key == 'type':
            continue

        if not isinstance(key, str) or not key.isidentifier():
            continue

        try:
            if isinstance(val, dict):
                child = dict_to_ast(val, f"{path}.{key}")
                if child is not None:
                    setattr(node, key, child)
            elif isinstance(val, list):
                converted = []
                for i, item in enumerate(val):
                    if isinstance(item, dict):
                        child = dict_to_ast(item, f"{path}.{key}[{i}]")
                        if child is not None:
                            converted.append(child)
                    elif not isinstance(item, dict):
                        converted.append(item)
                if converted:
                    setattr(node, key, converted)
            else:
                if not isinstance(val, dict):
                    setattr(node, key, val)
        except (AttributeError, TypeError):
            continue

    return node

def iter_children(node):
    if not isinstance(node, ast.AST):
        return
    try:
        for field, value in ast.iter_fields(node):
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, ast.AST):
                        yield item
            elif isinstance(value, ast.AST):
                yield value
    except Exception:
        pass


# ===== COLOR MAP & HELPERS ===========================================
COLOR_MAP = {
    'Module': '#4A90D9', 'FunctionDef': '#E94B3C', 'AsyncFunctionDef': '#E94B3C',
    'ClassDef': '#F5A623', 'If': '#7ED321', 'For': '#7ED321', 'AsyncFor': '#7ED321',
    'While': '#7ED321', 'Return': '#9B59B6', 'Call': '#1ABC9C', 'Assign': '#3498DB',
    'AugAssign': '#3498DB', 'AnnAssign': '#3498DB', 'Name': '#BDC3C7',
    'Constant': '#ECF0F1', 'Import': '#E67E22', 'ImportFrom': '#E67E22',
    'Expr': '#95A5A6', 'Compare': '#16A085', 'BoolOp': '#27AE60', 'BinOp': '#2980B9',
    'Attribute': '#8E44AD', 'Subscript': '#C0392B', 'List': '#D35400',
    'Dict': '#D35400', 'Tuple': '#D35400', 'Try': '#F39C12', 'With': '#1ABC9C',
    'Delete': '#E74C3C', 'Global': '#7F8C8D', 'Nonlocal': '#7F8C8D',
    'Pass': '#BDC3C7', 'Break': '#E74C3C', 'Continue': '#E74C3C',
    'Raise': '#C0392B', 'Assert': '#F39C12', 'Yield': '#9B59B6',
    'YieldFrom': '#9B59B6', 'Lambda': '#E94B3C', 'ExceptHandler': '#F39C12',
    'alias': '#E67E22', 'arguments': '#95A5A6', 'keyword': '#95A5A6',
}
DEFAULT_COLOR = '#DDEEFF'
LIGHT_NODES = {'#BDC3C7', '#ECF0F1', '#DDEEFF', '#95A5A6', '#7F8C8D'}

def _node_label(node):
    try:
        node_type = getattr(node, '_type_name', None) or type(node).__name__
        label = node_type
        name = getattr(node, 'name', None)
        nid = getattr(node, 'id', None)
        attr = getattr(node, 'attr', None)

        if node_type == 'alias':
            n = getattr(node, 'name', '')
            a = getattr(node, 'asname', '')
            label = f"alias\n{n}" + (f" as {a}" if a else "")
        elif node_type in ('Import', 'ImportFrom'):
            mod = getattr(node, 'module', '')
            label = f"{node_type}" + (f"\n{mod}" if mod else "")
        elif name and isinstance(name, str):
            label = f"{node_type}\n{str(name)[:14]}"
        elif nid and isinstance(nid, str):
            label = f"{node_type}\n{str(nid)[:14]}"
        elif attr and isinstance(attr, str):
            label = f"{node_type}\n.{str(attr)[:12]}"
        elif node_type == 'Constant':
            raw = getattr(node, 'value', '')
            label = f"Const\n{str(raw)[:12]}"
        elif node_type == 'BinOp':
            op_node = getattr(node, 'op', None)
            op_name = type(op_node).__name__ if op_node else ''
            label = f"BinOp\n{op_name}"
        elif node_type == 'BoolOp':
            op_node = getattr(node, 'op', None)
            op_name = type(op_node).__name__ if op_node else ''
            label = f"BoolOp\n{op_name}"
        elif node_type == 'Compare':
            ops = getattr(node, 'ops', [])
            ops_str = ','.join(type(o).__name__ for o in ops[:2]) if ops else ''
            label = f"Cmp\n{ops_str}"
        return label
    except Exception:
        return type(node).__name__


# ===== FUNGSI VISUALISASI ============================================
def visualize_ast_matplotlib(tree, nim, modul, filename, output_dir, max_nodes=MAX_NODES_VIS, max_depth=MAX_DEPTH_VIS):
    try:
        if not MATPLOTLIB_AVAILABLE:
            raise ImportError("Matplotlib tidak tersedia")

        positions = {}
        labels_map = {}
        node_types = {}
        edges = []
        counter = [0]
        node_count = [0]
        x_counter = [0]

        def build_graph(node, parent_id=None, depth=0):
            if node_count[0] >= max_nodes or depth > max_depth:
                return None
            if not isinstance(node, ast.AST):
                return None

            node_type = getattr(node, '_type_name', None) or type(node).__name__
            current_id = counter[0]
            counter[0] += 1
            node_count[0] += 1

            label = _node_label(node)
            children = list(iter_children(node))
            children_ids = []

            for child in children:
                cid = build_graph(child, current_id, depth + 1)
                if cid is not None:
                    children_ids.append(cid)
                    edges.append((current_id, cid))

            if children_ids:
                xs = [positions[cid][0] for cid in children_ids]
                my_x = (min(xs) + max(xs)) / 2
            else:
                my_x = x_counter[0]
                x_counter[0] += 1

            positions[current_id] = (my_x, -depth)
            labels_map[current_id] = label
            node_types[current_id] = node_type
            return current_id

        build_graph(tree)

        if not positions:
            return None, "Tidak ada node yang berhasil di-layout"

        n_leaves = max(x_counter[0], 1)
        fig_w = max(14, n_leaves * 0.9)
        fig, ax = plt.subplots(figsize=(fig_w, 10))
        ax.set_facecolor('#F8F9FA')
        fig.patch.set_facecolor('#F8F9FA')

        for (src, dst) in edges:
            x1, y1 = positions[src]
            x2, y2 = positions[dst]
            ax.plot([x1, x2], [y1, y2], color='#AAAAAA', lw=0.6, alpha=0.6, zorder=1)

        for nid, (x, y) in positions.items():
            node_type = node_types[nid]
            color = COLOR_MAP.get(node_type, DEFAULT_COLOR)
            fontcolor = '#FFFFFF' if color not in LIGHT_NODES else '#333333'
            ax.scatter(x, y, s=320, c=color, zorder=3, edgecolors='#334455', linewidths=0.7)
            ax.text(x, y, labels_map[nid], ha='center', va='center', fontsize=5.5, color=fontcolor, zorder=4, multialignment='center')

        total_nodes = node_count[0]
        ax.set_title(f"AST Tree — {nim} / {modul} / {filename}  [{total_nodes} node ditampilkan]", fontsize=10, fontweight='bold', pad=8)
        ax.axis('off')
        plt.tight_layout()

        out_subdir = os.path.join(output_dir, nim, modul)
        os.makedirs(out_subdir, exist_ok=True)
        out_path = os.path.join(out_subdir, filename.replace('.py', '.png'))
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        return out_path, None
    except Exception as e:
        return None, traceback.format_exc()

def visualize_ast_graphviz(tree, nim, modul, filename, output_dir, max_nodes=MAX_NODES_VIS, max_depth=MAX_DEPTH_VIS):
    try:
        if not GRAPHVIZ_AVAILABLE:
            raise ImportError("Graphviz tidak tersedia")
        
        import graphviz
        dot = graphviz.Digraph(comment=f'AST {nim}/{modul}/{filename}')
        dot.attr(rankdir='TB', size='18,14', dpi='150')
        dot.attr('node', shape='box', style='filled', fontname='Helvetica', fontsize='9', width='0.5', height='0.35')

        counter = [0]
        node_count = [0]

        def add_node(node, parent_id=None, depth=0):
            if node_count[0] >= max_nodes or depth > max_depth:
                return
            if not isinstance(node, ast.AST):
                return

            node_type = getattr(node, '_type_name', None) or type(node).__name__
            current_id = str(counter[0])
            counter[0] += 1
            node_count[0] += 1

            label = _node_label(node)
            color = COLOR_MAP.get(node_type, DEFAULT_COLOR)
            fontcolor = '#FFFFFF' if color not in LIGHT_NODES else '#333333'

            dot.node(current_id, label=label, fillcolor=color, fontcolor=fontcolor)
            if parent_id is not None:
                dot.edge(parent_id, current_id)

            for child in iter_children(node):
                add_node(child, current_id, depth + 1)

        add_node(tree)

        out_subdir = os.path.join(output_dir, nim, modul)
        os.makedirs(out_subdir, exist_ok=True)
        out_base = os.path.join(out_subdir, filename.replace('.py', ''))
        dot.render(out_base, format='png', cleanup=True)
        return out_base + '.png', None
    except Exception as e:
        return None, traceback.format_exc()

def visualize_ast(tree, nim, modul, filename, output_dir):
    if GRAPHVIZ_AVAILABLE:
        out_path, err = visualize_ast_graphviz(tree, nim, modul, filename, output_dir)
        if err is None:
            return out_path, None
        if MATPLOTLIB_AVAILABLE:
            return visualize_ast_matplotlib(tree, nim, modul, filename, output_dir)
        else:
            return None, f"Graphviz gagal dan Matplotlib tidak tersedia: {err}"
    elif MATPLOTLIB_AVAILABLE:
        return visualize_ast_matplotlib(tree, nim, modul, filename, output_dir)
    else:
        return None, "Graphviz dan Matplotlib tidak tersedia"


# ===== EKSEKUSI UTAMA ================================================
if len(all_ast_files) > 0:
    vis_success = 0
    vis_failed  = 0
    vis_skipped = 0
    
    visual_start = time.time()
    runtime_batch = []

    print(f"{'VISUALISASI AST (BACA JSON → PNG)':^70}")
    print("=" * 70)
    
    # tqdm.auto akan merender progress bar di atas (sebagai UI/HTML widget di Jupyter) 
    # dan log error yang dicetak akan berjejer rapi ke bawah.
    for ast_json_path, nim, modul, filename in tqdm(all_ast_files, desc="Visualisasi AST", unit="file"):
        file_start_time = time.time()
        file_id = os.path.splitext(filename)[0]
        try:
            with open(ast_json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            ast_dict = data.get('ast')
            if not ast_dict or not isinstance(ast_dict, dict):
                vis_skipped += 1
                continue

            tree = dict_to_ast(ast_dict)
            if tree is None:
                vis_skipped += 1
                continue

            out_path, err = visualize_ast(tree, nim, modul, filename, OUTPUT_VIS_DIR)
            if err:
                vis_failed += 1
                # Cetak log menggunakan standard print atau tqdm.write
                print(f"❌ [GAGAL] {nim}/{modul}/{filename}\n   └─ Error: {err[:150]}...")
            else:
                vis_success += 1
                file_runtime = time.time() - file_start_time

                runtime_batch.append({
                    "nim": nim,
                    "modul": modul,
                    "file": filename,
                    "file_id": file_id,
                    "value": file_runtime
                })

        except json.JSONDecodeError as e:
            vis_failed += 1
            print(f"❌ [GAGAL JSON] {nim}/{modul}/{filename}\n   └─ Error: {str(e)[:150]}")
        except Exception as e:
            vis_failed += 1
            print(f"❌ [GAGAL EXCEPTION] {nim}/{modul}/{filename}\n   └─ Error: {str(e)[:150]}")

    visual_total_time = time.time() - visual_start
    # Registrasi metrik mikro durasi pemrosesan grafik gambar/plot AST
    save_bulk_detail_metrics(RESULTS['METRICS'], "detail_runtime", runtime_batch, stage_name="AST Visualization")
    # Amalgamasi ringkasan makro kumulatif performa tahap 'AST Visualization'
    save_stage_evaluation(
        report_path=RESULTS['METRICS'],
        stage_name="AST Visualization",
        total_files=vis_success,
        execution_time=visual_total_time
    )
    
    display(HTML(f"<h3>RINGKASAN HASIL VISUALISASI AST</h3>"))
    print(f"{'Waktu Eksekusi':<10}: {visual_total_time:.2f} seconds")
    print(f"{'Berhasil':<10}: {vis_success}")
    print(f"{'Gagal':<10}: {vis_failed}")
    print(f"{'Skip':<10}: {vis_skipped}")
    print(f"{'Total':<10}: {len(all_ast_files)}")
    print(f"{'Output':<10}: {OUTPUT_VIS_DIR}")
    print("=" * 70)

else:
    print("⚠️  Tidak ada file AST untuk divisualisasi")

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Proses visualisasi selesai pada {timestamp}")

⚠️  Graphviz tidak tersedia, fallback ke matplotlib
✓ Matplotlib tersedia

VALIDASI: VISUALISASI AST
✓ Input folder  : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_AST
✓ Output folder : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06a_AST_visual
✓ Total AST JSON: 2290 file
✓ Siap visualisasi 2290 file AST

                  VISUALISASI AST (BACA JSON → PNG)                   


Visualisasi AST:   0%|          | 0/2290 [00:00<?, ?file/s]

TypeError: save_stage_evaluation() missing 1 required positional argument: 'total_size_kb'

## 2. Transformasi AST ke Graph

### Transform AST -> Graph

In [10]:
import networkx as nx
import pickle

# FUNCTION HELPER - GRAPH CONSTRUCTION (PICKLE VERSION)
# ==============================================================================

def save_log(nim, modul, file, n_nodes=0, n_edges=0, status="SUCCESS", message=""):
    return {
        "nim": nim, "modul": modul, "file": file,
        "jumlah_node_graf": n_nodes, "jumlah_edge_graf": n_edges,
        "status": status, "keterangan": message
    }

def get_rich_label(ast_node):
    node_type = ast_node.get("type", "Unknown")
    
    # 1. Operasi Biner (misal: Add, Sub, Mult)
    if node_type == "BinOp" and "op" in ast_node and isinstance(ast_node["op"], dict):
        return f"BinOp_{ast_node['op'].get('type', '')}"
        
    # 2. Operasi Perbandingan (misal: Eq, Gt, Lt)
    elif node_type == "Compare" and "ops" in ast_node and isinstance(ast_node["ops"], list):
        ops_labels = [op.get("type", "") for op in ast_node["ops"] if isinstance(op, dict)]
        if ops_labels:
            return f"Compare_{'_'.join(ops_labels)}"
            
    # 3. Operasi Boolean (misal: And, Or)
    elif node_type == "BoolOp" and "op" in ast_node and isinstance(ast_node["op"], dict):
        return f"BoolOp_{ast_node['op'].get('type', '')}"
        
    # 4. Tipe Konstanta (membedakan int, float, str tanpa mengambil nilainya)
    elif node_type == "Constant" and "value" in ast_node:
        val_type = type(ast_node["value"]).__name__
        return f"Constant_{val_type}"
        
    # Jika tidak masuk kategori di atas, kembalikan tipe aslinya
    return node_type

def transform_ast_to_nx(ast_node, G=None, parent_id=None):
    if G is None: 
        G = nx.Graph()
        
    current_id = G.number_of_nodes()
    
    # Ambil tipe AST (misal: 'For', 'Assign', 'BinOp')
    raw_feature = get_rich_label(ast_node)
    node_feature = str(raw_feature)
    
    # Masukkan ke graf menggunakan key 'feature'
    G.add_node(current_id, feature=node_feature)
    
    if parent_id is not None: 
        G.add_edge(parent_id, current_id)
        
    for field, value in ast_node.items():
        if field == "type": continue
        
        if field in ["op", "ops"] and ast_node.get("type") in ["BinOp", "Compare", "BoolOp", "UnaryOp"]:
            continue
        
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict) and "type" in item:
                    transform_ast_to_nx(item, G, current_id)
                    
        elif isinstance(value, dict) and "type" in value:
            transform_ast_to_nx(value, G, current_id)
            
    return G

# SAVE GRAPH AS PICKLE
def save_graph_pkl(G, metadata, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "wb") as f:
        pickle.dump({"metadata": metadata, "graph": G}, f)
        
def normalize_graph(G):
    return nx.convert_node_labels_to_integers(G, first_label=0)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Helper functions untuk graph construction dijalankan pada {timestamp}")

✅ Helper functions untuk graph construction dijalankan pada 2026-05-26 03:01:56


In [11]:
# RUN & EXECUTION
# ==============================================================================

INPUT_DIR = DIRS['AST']   # Folder JSON AST
OUTPUT_DIR = DIRS['GRAPH'] # Folder baru untuk Dataset Graph (.pkl per file)

# Mengumpulkan file JSON AST
all_ast_files = []
for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith(".json"):
            all_ast_files.append(os.path.join(root, file))

logs = []
success = 0
failed = 0

graph_start = time.time()
runtime_batch = []
size_batch = []

print("="*60)
print(f"{' CONSTRUCT GRAPH DATASET ':^60}")
print("-"*60)
print(f"{'Dataset input':<30}: {INPUT_DIR}")
print(f"{'Total File input':<30}: {len(all_ast_files)}")

for file_path in tqdm(all_ast_files, desc="Transforming to Graph", unit="file"):
    file_start_time = time.time()
    # 1. Extract Metadata
    rel_path = os.path.relpath(file_path, INPUT_DIR)
    parts = rel_path.split(os.sep)
    nim, modul, filename = parts[0], parts[1], os.path.basename(file_path)
    file_id = os.path.splitext(filename)[0]
    
    try:
        # 2. Load AST & Transform
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        G = transform_ast_to_nx(data['ast'])
        G = normalize_graph(G)
        
        # 3. Save as individual .pkl
        output_path = os.path.join(OUTPUT_DIR, rel_path.replace(".json", ".pkl"))
        meta = {"nim": nim, "modul": modul, "file": filename, "jumlah_nodes":G.number_of_nodes(), "jumlah_edge":G.number_of_edges()}
        
        save_graph_pkl(G, meta, output_path)
        
        file_runtime = time.time() - file_start_time
        graph_size = get_path_size(output_path, unit='kb')
        # Enkapsulasi basis data identitas berkas ke dalam objek kamus tunggal
        base_data = {"nim": nim, "modul": modul, "file": filename, "file_id": file_id}
        # Alokasi metrik paralel menggunakan teknik dictionary unpacking
        runtime_batch.append({**base_data, "value": file_runtime})
        size_batch.append({**base_data, "value": graph_size})
        
        logs.append(save_log(nim, modul, filename, G.number_of_nodes(), G.number_of_edges()))
        success += 1
        
    except Exception as e:
        logs.append(save_log(nim, modul, filename, status="FAILED", message=str(e)))
        failed += 1

print(f"\nDataset Graph (Pickle) disimpan di: {OUTPUT_DIR}")
print(f"{' PROSES TRANSFORM SELESAI ':=^60}")

# SUMMARY & REPORTING
# ==============================================================================
df_logs = pd.DataFrame(logs)
df_success = df_logs[df_logs['status'] == 'SUCCESS']

# Simpan Report
report_path = RESULTS['CONSTRUCT_GRAPH']
os.makedirs(os.path.dirname(report_path), exist_ok=True)
df_logs.to_excel(report_path, index=False)

total_input = len(all_ast_files)
total_processed = len(df_logs)

graph_total_time = time.time() - graph_start
# Registrasi metrik mikro (runtime & kapasitas berkas) untuk tahap Graph Construction
save_bulk_detail_metrics(RESULTS['METRICS'], "detail_runtime", runtime_batch, stage_name="Graph Construction")
save_bulk_detail_metrics(RESULTS['METRICS'], "detail_size", size_batch, stage_name="Graph Construction")
# Pencatatan ringkasan evaluasi makro kumulatif direktori target GRAPH
save_stage_evaluation(
    report_path=RESULTS['METRICS'],
    stage_name="Graph Construction",
    total_files=count_all_files(DIRS['GRAPH'])['total_files'],
    total_size_kb=get_path_size(DIRS['GRAPH'], unit='kb'),
    execution_time=graph_total_time
)

display(HTML(f"<h3>RINGKASAN HASIL KONSTRUKSI GRAPH</h3>"))
print("-" * 75)

print(f"{'Total File Input':<35}: {total_input}")
print(f"{'Total Diproses':<35}: {total_processed}")
print("-" * 75)
print(f"{'Berhasil (SUCCESS)':<35}: {success}")
print(f"{'Gagal':<35}: {failed}")
print(f"{'Success Rate':<35}: {(success/len(all_ast_files)*100):.2f}%")
print("-" * 75)
print(f"{'Report Excel':<25}: {report_path}")
print(f"{'Dataset Graph':<25}: {OUTPUT_DIR}")

# Preview
print("\nPreview Data (10 baris pertama):")
display(df_logs.head(10))

# Timestamp
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada {timestamp}")

                  CONSTRUCT GRAPH DATASET                   
------------------------------------------------------------
Dataset input                 : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\06_AST
Total File input              : 2290


Transforming to Graph:   0%|          | 0/2290 [00:00<?, ?file/s]


Dataset Graph (Pickle) disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_Graph
================= PROSES TRANSFORM SELESAI =================


---------------------------------------------------------------------------
Total File Input                   : 2290
Total Diproses                     : 2290
---------------------------------------------------------------------------
Berhasil (SUCCESS)                 : 2290
Gagal                              : 0
Success Rate                       : 100.00%
---------------------------------------------------------------------------
Report Excel             : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\09b_Graph_report.xlsx
Dataset Graph            : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_Graph

Preview Data (10 baris pertama):


,nim,modul,file,jumlah_node_graf,jumlah_edge_graf,status,keterangan
0,2241720092,js02,p01.json,414,413,SUCCESS,
1,2241720092,js02,p02.json,113,112,SUCCESS,
2,2241720092,js02,p03.json,212,211,SUCCESS,
3,2241720092,js02,p04.json,142,141,SUCCESS,
4,2241720092,js02,tp.json,632,631,SUCCESS,
5,2241720092,js03,p01.json,498,497,SUCCESS,
6,2241720092,js03,p02.json,142,141,SUCCESS,
7,2241720092,js03,p03.json,224,223,SUCCESS,
8,2241720092,js03,p04.json,53,52,SUCCESS,
9,2241720092,js03,tp.json,1163,1162,SUCCESS,


Diproses pada 2026-05-26 03:02:45


### Building List of Graph

In [12]:
# FUNCTION HELPER - GRAPH2VEC INPUT PREPARATION
# ==============================================================================

def collect_pkl_files(input_dir):
    """Mengumpulkan semua path file .pkl dari direktori dataset graf."""
    pkl_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".pkl"):
                pkl_files.append(os.path.join(root, file))
    return pkl_files

def normalize_graph_for_karateclub(G):
    """
    Normalisasi wajib untuk algoritma Graph2Vec (Karateclub):
    1. Memastikan graf tidak berarah (Undirected).
    2. Memastikan ID node adalah integer berurutan yang dimulai dari 0.
    """
    # 1. Konversi ke Undirected Graph jika masih berarah
    if G.is_directed():
        G = G.to_undirected()
        
    # 2. Reset ID node agar berurutan (0, 1, 2, ...) 
    # tanpa menghilangkan atribut 'feature' yang sudah kita buat
    G = nx.convert_node_labels_to_integers(G, first_label=0)
    
    return G

def load_single_graph_data(filepath):
    """Membaca satu file .pkl yang berisi graf dan metadatanya."""
    try:
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        return data.get("graph"), data.get("metadata"), "SUCCESS", ""
    except Exception as e:
        return None, None, "FAILED", str(e)

def save_log(meta, n_nodes=0, n_edges=0, status="SUCCESS", message=""):
    return {
        "nim": meta.get("nim") if meta else None,
        "modul": meta.get("modul") if meta else None,
        "file": meta.get("file") if meta else None,
        "jumlah_node": n_nodes,
        "jumlah_edge": n_edges,
        "group_key": f"{meta.get('modul')}_{meta.get('file')}" if meta else None,
        "status": status,
        "keterangan": message
    }

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Helper functions untuk persiapan input Graph2Vec dijalankan pada {timestamp}")

✅ Helper functions untuk persiapan input Graph2Vec dijalankan pada 2026-05-26 03:02:45


In [14]:
# RUN & EXECUTION
# ==============================================================================
INPUT_DIR = DIRS['GRAPH']  # Folder tempat file .pkl per-file disimpan sebelumnya
OUTPUT_DATASET_PATH = DIRS['INPUT_GRAPH']
all_pkl_files = collect_pkl_files(INPUT_DIR)

grouped_graphs = {}
logs = []
success_count = 0
group_start = time.time()
runtime_batch = []
print("="*60)
print(f"{' PREPARING LIST OF GRAPHS FOR GRAPH2VEC ':^60}")
print("-"*60)
print(f"{'Input Directory':<30}: {INPUT_DIR}")
print(f"{'Total Files Found':<30}: {len(all_pkl_files)}")

for file_path in tqdm(all_pkl_files, desc="Consolidating Dataset", unit="file"):
    file_start_time = time.time()
    filename = os.path.basename(file_path)
    # 1. Load Data
    G, meta, status, msg = load_single_graph_data(file_path)
    
    if status == "SUCCESS" and G is not None:
        try:
            # 2. Normalisasi Graf untuk Karateclub
            G_norm = normalize_graph_for_karateclub(G)
            # 3. Masukkan ke List Utama (Wajib tersinkronisasi urutannya)
            key = (meta["modul"], meta["file"])
            if key not in grouped_graphs:
                grouped_graphs[key] = {
                    "graphs": [],
                    "metadata": []
                }

            grouped_graphs[key]["graphs"].append(G_norm)
            grouped_graphs[key]["metadata"].append(meta)
            
            logs.append(save_log(meta, n_nodes=G_norm.number_of_nodes(), n_edges=G_norm.number_of_edges(), status="SUCCESS", message="Added to group"))
            success_count += 1
            
            file_runtime = time.time() - file_start_time
            file_id = os.path.splitext(meta["file"])[0]
            runtime_batch.append({
                "nim": meta["nim"],
                "modul": meta["modul"],
                "file": meta["file"],
                "file_id": file_id,
                "value": file_runtime
            })
        except Exception as e:
            logs.append(save_log(meta, status="FAILED", message=f"Normalization Error: {str(e)}"))
    else:
        logs.append(save_log(None, status="FAILED", message=f"{filename} | Load Error: {msg}"))

# 4. Simpan List of Graphs dan Metadata
for (modul, file), data in grouped_graphs.items():
    modul_dir = os.path.join(OUTPUT_DATASET_PATH, modul)
    os.makedirs(modul_dir, exist_ok=True)
    output_path = os.path.join(modul_dir, f"{modul}_{file.replace('.py','')}.pkl")
    with open(output_path, "wb") as f:
        pickle.dump(data, f)

print(f"\nFinal Dataset disimpan di: {OUTPUT_DATASET_PATH}")
print(f"{' PROSES SELESAI ':-^60}")

# SUMMARY & REPORTING
# ==============================================================================
REPORT_PATH = RESULTS['LIST_GRAPH']
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
df_logs = pd.DataFrame(logs)

total_files = len(all_pkl_files)
success_rate = (success_count / total_files * 100) if total_files > 0 else 0

# Statistik grouping
total_groups = len(grouped_graphs)
group_sizes = [len(v["graphs"]) for v in grouped_graphs.values()]

avg_graph_per_group = sum(group_sizes) / total_groups if total_groups > 0 else 0
min_graph = min(group_sizes) if group_sizes else 0
max_graph = max(group_sizes) if group_sizes else 0

group_total_time = time.time() - group_start
save_bulk_detail_metrics(
    RESULTS['METRICS'],
    "detail_runtime",
    runtime_batch,
    stage_name="Graph Grouping"
)
save_stage_evaluation(
    report_path=RESULTS['METRICS'],
    stage_name="Graph Grouping",
    total_files=success_count,
    execution_time=group_total_time,
    total_size_kb=get_path_size(OUTPUT_DATASET_PATH, unit='kb')
)

display(HTML(f"<h3>RINGKASAN DATASET GRAPH2VEC (GROUPED)</h3>"))
print(f"{'Total File PKL Input':<40}: {total_files}")
print(f"{'Berhasil Diproses (SUCCESS)':<40}: {success_count}")
print(f"{'Gagal (FAILED)':<40}: {total_files - success_count}")
print(f"{'Success Rate':<40}: {success_rate:.2f}%")
print("-" * 75)
print(f"{'Total Group (modul-file)':<40}: {total_groups}")
print(f"{'Rata-rata graph per group':<40}: {avg_graph_per_group:.2f}")
print(f"{'Minimum graph per group':<40}: {min_graph}")
print(f"{'Maximum graph per group':<40}: {max_graph}")
print("-" * 75)

# VALIDASI GROUP
valid_groups = sum(1 for v in grouped_graphs.values() if len(v["graphs"]) >= 2)
invalid_groups = total_groups - valid_groups
print(f"{'Group valid (≥2 graph)':<40}: {valid_groups}")
print(f"{'Group tidak valid (<2 graph)':<40}: {invalid_groups}")
print("-" * 75)
print(f"{'Dataset Output Directory':<40}: {OUTPUT_DATASET_PATH}")

# Preview
print("\nPreview Group (5 pertama):")
preview_data = []
for i, ((modul, file), v) in enumerate(grouped_graphs.items()):
    if i >= 5:
        break
    preview_data.append({
        "modul": modul,
        "file": file,
        "jumlah_graph": len(v["graphs"])
    })

df_preview = pd.DataFrame(preview_data)
display(df_preview)

# SAVE TO EXCEL (LOG + VALIDATION)
# Group summary
group_records = []
for (modul, file), data in grouped_graphs.items():
    n_graph = len(data["graphs"])
    group_records.append({
        "modul": modul,
        "file": file,
        "jumlah_graph": n_graph,
        "status_group": "VALID" if n_graph >= 2 else "INVALID"
    })
df_group = pd.DataFrame(group_records)
# Metadata detail
meta_records = []
for (modul, file), data in grouped_graphs.items():
    for m in data["metadata"]:
        meta_records.append({
            "modul": modul,
            "file": file,
            "nim": m.get("nim"),
            "filename": m.get("file")
        })

df_meta = pd.DataFrame(meta_records)

# Validation summary
df_validation = pd.DataFrame([{
    "total_files": total_files,
    "success": success_count,
    "failed": total_files - success_count,
    "success_rate (%)": round(success_rate, 2),
    "total_groups": total_groups,
    "valid_groups": valid_groups,
    "invalid_groups": invalid_groups,
    "avg_graph_per_group": round(avg_graph_per_group, 2),
    "min_graph": min_graph,
    "max_graph": max_graph
}])

# Save Excel
with pd.ExcelWriter(REPORT_PATH, engine="openpyxl") as writer:
    df_logs.to_excel(writer, sheet_name="logs", index=False)
    df_group.to_excel(writer, sheet_name="group_summary", index=False)
    df_meta.to_excel(writer, sheet_name="metadata", index=False)
    df_validation.to_excel(writer, sheet_name="validation", index=False)

    # Auto width
    for ws in writer.sheets.values():
        for col in ws.columns:
            max_len = 0
            col_letter = col[0].column_letter
            for cell in col:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max_len + 2, 50)

print(f"\n📊 Report Excel disimpan di: {REPORT_PATH}")

print("\n" + "=" * 75)
timestamp_finish = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Diproses pada: {timestamp_finish}")

           PREPARING LIST OF GRAPHS FOR GRAPH2VEC           
------------------------------------------------------------
Input Directory               : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\07_Graph
Total Files Found             : 2290


Consolidating Dataset:   0%|          | 0/2290 [00:00<?, ?file/s]


Final Dataset disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph2vec_Input
---------------------- PROSES SELESAI ----------------------


Total File PKL Input                    : 2290
Berhasil Diproses (SUCCESS)             : 2290
Gagal (FAILED)                          : 0
Success Rate                            : 100.00%
---------------------------------------------------------------------------
Total Group (modul-file)                : 49
Rata-rata graph per group               : 46.73
Minimum graph per group                 : 21
Maximum graph per group                 : 53
---------------------------------------------------------------------------
Group valid (≥2 graph)                  : 49
Group tidak valid (<2 graph)            : 0
---------------------------------------------------------------------------
Dataset Output Directory                : D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\dataset(2)\08_Graph2vec_Input

Preview Group (5 pertama):


,modul,file,jumlah_graph
0,js02,p01.json,52
1,js02,p02.json,49
2,js02,p03.json,49
3,js02,p04.json,49
4,js02,tp.json,49



📊 Report Excel disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\output(2)\09c_List_Graph_report.xlsx

Diproses pada: 2026-05-26 03:13:11
